7 - QUERYING PLANET DATA

This notebook queries, orders, and downloads Planet imagery for paired GEDI footprints, then filters the GEDI pair dataset to retain only pairs with usable Planet coverage.

Workflow:
- Load the GEDI pair GeoPackage and build separate pre-fire and post-fire footprint GeoDataFrames with acquisition dates and pairing CRS information.
- Authenticate to Planet and define search rules that look for nearby scenes within expanding time windows and cloud-cover thresholds.
- Search for candidate Planet scenes for each footprint, rank them by temporal closeness to the GEDI target date, and keep the best pre and post image per pair. (Used Planet SCA in this part)
- Retain only pairs where both footprints have matching Planet scenes, and save the selected scene list for ordering.
- Create buffered AOIs around each footprint, submit clipped Planet image orders through the Planet SDK, and track order status in a persistent JSON log.
- Validate the order log, download completed orders into a project storage folder, and record downloads in a separate download log.
- Identify failed orders, especially access-denied cases, select alternate candidate scenes for retry, and prepare a retry table.
- Remove GEDI pairs where one or both Planet image requests ultimately failed, and prepare a filtered GEDI dataset for downstream analysis.

In [65]:
import pandas as pd
import geopandas as gpd
import math
import time
import json
import glob
import xarray as xr
import uuid
import matplotlib.pyplot as plt
import json
import tempfile
import os
import requests
import asyncio

from collections import Counter
from shapely.validation import explain_validity
from shapely.geometry import mapping
from tqdm.auto import tqdm
from shapely import wkt
from pathlib import Path
from datetime import timedelta
from datetime import datetime, timezone
from uuid import uuid4

import planet
from planet import Auth
from planet import Session, OrdersClient
from planet import data_filter

In [24]:
# Authenticate to Planet
auth = Auth.from_user_default_session()

if not auth.is_initialized():
    raise RuntimeError(
        "Not authenticated. Run:\n\n    planet auth login\n"
    )



In [25]:
#Load pairs
pairs = gpd.read_file("GEDI_Pairs_Final.gpkg")


# Define a root folder in Drive for Planet images
drive_planet = Path("/Users/user123/Library/CloudStorage/GoogleDrive-misantos@uw.edu/My Drive/GEDI/Planet_downloads")
drive_planet.mkdir(parents=True, exist_ok=True)

print("Saving Planet imagery to:", drive_planet)


Saving Planet imagery to: /Users/user123/Library/CloudStorage/GoogleDrive-misantos@uw.edu/My Drive/GEDI/Planet_downloads


In [26]:
# Pre-fire footprint queries (geometry_1)
pre_query = gpd.GeoDataFrame(
    {   "pair_uid": pairs["pair_uid"],
        "time_1": pairs["time_1"],
        "pairing_crs": pairs["pairing_crs"]
    },
    geometry=pairs["geometry_1"].apply(
        lambda x: wkt.loads(x) if pd.notna(x) else None
    ),
    crs=pairs.crs
)

# Post-fire footprint queries (geometry_2)
post_query = gpd.GeoDataFrame(
    {   "pair_uid": pairs["pair_uid"],
        "time_2": pairs["time_2"],
        "pairing_crs": pairs["pairing_crs"]
    },
    geometry=pairs["geometry_2"].apply(
        lambda x: wkt.loads(x) if pd.notna(x) else None
    ),
    crs=pairs.crs
)


pre_query["time_1"] = pd.to_datetime(pre_query["time_1"], utc=True)
post_query["time_2"] = pd.to_datetime(post_query["time_2"], utc=True)


In [ ]:
""""

#Exporting the pre and post data as GeoJSONs

tmp_dir = Path("Footprints")
tmp_dir.mkdir(exist_ok=True)

pre_path  = tmp_dir / "pre.geojson"
post_path = tmp_dir / "post.geojson"

pre_query.to_file(pre_path, driver="GeoJSON")
post_query.to_file(post_path, driver="GeoJSON")
print("GeoJSONs written successfully")

In [ ]:
################## TO GET THE IDs OF THE REQUIRED IMAGES ############################

# Function to search for the Search Planet imagery for the acquisition closest in time to target_date
# over a given footprint geometry (GeoJSON file path).
# Returns a dictionary with metadata or None values if nothing is found.

# Use Search Tiers to change the filtering parameters 
SEARCH_TIERS = [
    {"days": 3, "cloud": 0.05},
    {"days": 7, "cloud": 0.10},
    {"days": 14, "cloud": 0.20},
    {"days": 30, "cloud": 0.30},
]


def search_planet_all_matches(
    api_key,
    geom_geojson_path,
    target_date,
    pair_uid,
    footprint_type
):
    
    #  Guard for missing dates   
    if pd.isna(target_date):
        return []

    # Normalize target date: explicit timezone (UTC)    
    target_date = pd.to_datetime(target_date, utc=True)
    all_rows = []

    for tier_i, tier in enumerate(SEARCH_TIERS, start=1):
        start = (target_date - timedelta(days=tier["days"])).isoformat()
        end   = (target_date + timedelta(days=tier["days"])).isoformat()

        filters = ps.search.combine_filters([
            ps.search.make_date_range_filter(start, end),
            ps.search.make_geometry_filter_from_geojson(geom_geojson_path),
            ps.search.make_cloud_cover_filter(tier["cloud"])
        ])

        # Planet search       
        results = ps.search.search(api_key, filters)
        
        # Deal with Planet returning None or empty dataframe
        if results is None or len(results) == 0:
            continue
            
        # Optional overlap filter 
        results = results[results["overlap_percentage"] >= 50]
        if len(results) == 0:
            continue
            
        # Normalize acquisition timezones (UTC) 
        results["acquired"] = pd.to_datetime(results["acquired"], utc=True)

        # Temporal distance        
        results["days_from_target"] = (
            results["acquired"] - target_date
        ).abs().dt.days

        for _, r in results.iterrows():
            all_rows.append({
                "pair_uid": pair_uid,                 # Original pair UID
                "footprint_type": footprint_type,     # pre or post
                "planet_item_id": r["id"],
                "acquired_date": r["acquired"],
                "days_from_target": int(r["days_from_target"]),
                "cloud_cover": r["cloud_cover"],
                "overlap_pct": r["overlap_percentage"],
                "search_tier": tier_i
            })

        break

    return all_rows


In [27]:
# Function to create every geometry and add it to a temporary file 

def write_temp_geojson(geom, tmp_dir):
    tmp_dir.mkdir(parents=True, exist_ok=True)

    geojson = {
        "type": "FeatureCollection",
        "features": [{
            "type": "Feature",
            "properties": {},
            "geometry": geom
        }]
    }

    tmp_path = tmp_dir / f"aoi_{uuid.uuid4().hex}.geojson"

    with open(tmp_path, "w") as f:
        json.dump(geojson, f)

    return tmp_path


In [ ]:
#Selct just the best images (closer to the target day)
best_planet = (
    planet_df
    .sort_values("days_from_target")
    .groupby(["pair_uid", "footprint_type"])
    .head(1)
)

print(f"Total Planet scenes found: {len(best_planet)}")
#best_planet.to_csv("planet_images_best.csv", index=False)

In [ ]:
# Get only the footprints that returned images 

best_planet_complete = best_planet.groupby("pair_uid").filter(
    lambda g: g["footprint_type"].nunique() == 2
)


In [ ]:
# Count the amount of footprints 
counts = (
    best_planet_complete
    .groupby("pair_uid")["footprint_type"]
    .nunique()
)
counts.value_counts()

In [ ]:
best_planet = (best_planet.sort_values("pair_uid"))
best_planet.head()

In [ ]:
################## WITH THE PLANET ITEM IDs WE CAN NOW ORDER THE IMAGES ############
# We will use a buffer of the footprints to get an AOI and download only the clips that we need from each image


In [22]:
best_planet = pd.read_csv("planet_best_images_complete.csv")
print("Images to order:", len(best_planet))

download_df = best_planet.copy()
download_df.head()

Images to order: 7726


,pair_uid,footprint_type,planet_item_id,acquired_date,days_from_target,cloud_cover,overlap_pct,search_tier
0,0,post,20220430_132000_1010,2022-04-30 13:20:00.465788+00:00,22,0.04,100.0,4
1,0,pre,20200423_142308_84_106e,2020-04-23 14:23:08.847897+00:00,2,0.15,100.0,3
2,1,pre,20200920_125602_76_2257,2020-09-20 12:56:02.764738+00:00,0,0.03,100.0,1
3,1,post,20230117_131458_20_24a3,2023-01-17 13:14:58.206417+00:00,12,0.21,100.0,4
4,2,post,20230117_131458_20_24a3,2023-01-17 13:14:58.206417+00:00,12,0.21,100.0,4


In [29]:
# To solve problems with pairing_crs

def attach_pairing_crs(download_df, pre_query, post_query):
    crs_map = {}

    for _, row in pre_query.iterrows():
        crs_map[(row["pair_uid"], "pre")] = row["pairing_crs"]

    for _, row in post_query.iterrows():
        crs_map[(row["pair_uid"], "post")] = row["pairing_crs"]

    download_df = download_df.copy()
    download_df["pairing_crs"] = download_df.apply(
        lambda r: crs_map[(r["pair_uid"], r["footprint_type"])],
        axis=1
    )

    return download_df


download_df = attach_pairing_crs(download_df, pre_query, post_query)


def order_key(row):
    return f"{row['pair_uid']}_{row['footprint_type']}_{row['planet_item_id']}"


In [30]:
def atomic_json_dump(obj, path):
    path = Path(path)
    with tempfile.NamedTemporaryFile("w", dir=path.parent, delete=False) as tmp:
        json.dump(obj, tmp, indent=2)
        tmp_name = tmp.name
    os.replace(tmp_name, path)


In [31]:
# Function to create the buffers
def write_buffered_aoi_geojson_with_pairing_crs(
    geometry,
    pairing_crs,
    buffer_m,
    out_path
):

    gdf = gpd.GeoDataFrame(
        geometry=[geometry],
        crs=4326
    )

    # Project → buffer → back to WGS84
    gdf_proj = gdf.to_crs(pairing_crs)
    gdf_proj["geometry"] = gdf_proj.buffer(buffer_m)
    gdf_wgs84 = gdf_proj.to_crs(4326)

    geojson = {
        "type": "FeatureCollection",
        "features": [{
            "type": "Feature",
            "properties": {},
            "geometry": mapping(gdf_wgs84.iloc[0].geometry)
        }]
    }

    with open(out_path, "w") as f:
        json.dump(geojson, f)

    return out_path


In [10]:
# CREATE AN ORDER_LOG TO KEEP RECORD OF WHAT WAS ALREADY ORDERED AND THE URLs 
# VERY VERY careful with this one. It could delete the entire order_log. 
#To use in case of "emergency" 

"""
ORDER_LOG_PATH = Path("planet_order_log.json")

order_log = {}

with open(ORDER_LOG_PATH, "w") as f:
    json.dump(order_log, f, indent=2)

print("Initialized clean planet_order_log.json")



Initialized clean planet_order_log.json


In [32]:
# Start the session

session = Session()
orders_client = OrdersClient(session)


In [51]:
#################  ORDER THE IMAGES (SCALED) #################

# Footprint -> radius = 12.5 m (GEDI size) + 10.2 m (geolocation radius) = 22.7
# To get the immediate landscape + the pairing footprint -> 100m , therefore, buffer = 77.3

BUFFER_M = 77.3                     # meters
PLANET_SLEEP_S = 2.0                # delay (important so Planet has time to process the request)
MAX_ORDERS_PER_RUN = 1              

tmp_aoi_dir = Path("/tmp/planet_order_aois")
tmp_aoi_dir.mkdir(parents=True, exist_ok=True)

ORDER_LOG_PATH = Path("planet_order_log.json")


# -------------------- LOAD / INIT ORDER LOG --------------------

if ORDER_LOG_PATH.exists():
    order_log = json.load(open(ORDER_LOG_PATH, "r"))
else:
    order_log = {}

# Track attempts vs successes separately
attempted_this_run = 0
submitted_this_run = 0

# -------------------- ATOMIC JSON WRITE --------------------

def atomic_json_dump(obj, path):
    path = Path(path)
    with tempfile.NamedTemporaryFile("w", dir=path.parent, delete=False) as tmp:
        json.dump(obj, tmp, indent=2)
        tmp_name = tmp.name
    os.replace(tmp_name, path)



# -------------------- AUTH CHECK --------------------

auth = Auth.from_user_default_session()
if not auth.is_initialized():
    raise RuntimeError(
        "Planet authentication missing.\n\n"
        "Run once in a terminal:\n\n"
        "    planet auth login\n"
    )

# -------------------- MAIN LOOP ----------------------

async def submit_order(sess, order_name, item_id, aoi_geometry):
    orders = OrdersClient(sess)

    order_request = {
        "name": order_name,
        "source_type": "scenes",
        "products": [
            {
                "item_ids": [item_id],
                "item_type": "PSScene",
                "product_bundle": "analytic_udm2",
            }
        ],
        "tools": [
            {
                "clip": {
                    "aoi": aoi_geometry
                }
            }
        ]
    }

    order = await orders.create_order(order_request)
    return order["id"]


# -------------------- MAIN ASYNC LOOP --------------------

async def submit_orders():

    global attempted_this_run, submitted_this_run

    async with Session(auth=auth) as sess:

        for _, row in tqdm(
            retry_df.iterrows(),                     ######## CHANGE INPUT DATA HEEEEERE !!!!!
            total=len(retry_df),                     ######## AND HERE TOO !!!!!
            desc="Submitting Planet orders (Planet SDK)"
        ):

            # Stop ONLY based on attempts
            if attempted_this_run >= MAX_ORDERS_PER_RUN:
                break

            key = f"{row['pair_uid']}_{row['footprint_type']}"

            # Skip anything already handled

            if key in order_log and order_log[key]["status"] != "failed":
                continue


            # Count this attempt
            attempted_this_run += 1

            try:
                # Select correct footprint
                src = pre_query if row["footprint_type"] == "pre" else post_query
                rec = src[src["pair_uid"] == row["pair_uid"]].iloc[0]

                print("AOI validity:", explain_validity(rec.geometry))

                # Write buffered AOI
                aoi_path = tmp_aoi_dir / f"aoi_{key}.geojson"
                write_buffered_aoi_geojson_with_pairing_crs(
                    geometry=rec.geometry,
                    pairing_crs=rec["pairing_crs"],
                    buffer_m=BUFFER_M,
                    out_path=aoi_path
                )

                with open(aoi_path) as f:
                    aoi_geometry = json.load(f)["features"][0]["geometry"]

                # Build unique, readable order name
                order_name = (
                    f"gedi_{row['pair_uid']}_"
                    f"{row['footprint_type']}_"
                    f"{row['planet_item_id']}"
                )

                # Submit order
                order_id = await submit_order(
                    sess,
                    order_name,
                    row["planet_item_id"],
                    aoi_geometry
                )

                # Record success
                order_log[key] = {
                    "pair_uid": row["pair_uid"],
                    "footprint_type": row["footprint_type"],
                    "planet_item_id": row["planet_item_id"],
                    "order_name": order_name,
                    "order_id": order_id,
                    "status": "submitted"
                }

                atomic_json_dump(order_log, ORDER_LOG_PATH)

                submitted_this_run += 1
                time.sleep(PLANET_SLEEP_S)

            except Exception as e:
                msg = str(e)

                if "no access to assets" in msg:
                    failure_category = "access_denied"
                else:
                    failure_category = "other"

                order_log[key] = {
                    "pair_uid": row["pair_uid"],
                    "footprint_type": row["footprint_type"],
                    "planet_item_id": row["planet_item_id"],
                    "status": "failed",
                    "failure_category": failure_category,
                    "error_message": msg
                }

                atomic_json_dump(order_log, ORDER_LOG_PATH)
                print(f"Order failed for {key}: {msg}")

    print(f"Attempts this run: {attempted_this_run}")
    print(f"Successful submissions this run: {submitted_this_run}")
    print(f"Total records in order log: {len(order_log)}")




await submit_orders()


Submitting Planet orders (Planet SDK):   0%|          | 0/1 [00:00<?, ?it/s]

AOI validity: Valid Geometry
Attempts this run: 1
Successful submissions this run: 1
Total records in order log: 7726


In [ ]:

order_log = json.load(open("planet_order_log.json"))

list(order_log.items())[:2]


In [64]:
#################  VALIDATE ORDER LOG #################

ORDER_LOG_PATH = Path("planet_order_log.json")

order_log = json.load(open(ORDER_LOG_PATH, "r"))

print(f"Total entries in order_log: {len(order_log)}")

# ---- 1. Required fields check ----
required_fields = {"pair_uid", "footprint_type", "planet_item_id", "status"}

missing_fields = {
    k: required_fields - set(v.keys())
    for k, v in order_log.items()
    if not required_fields.issubset(v.keys())
}

print("Entries missing required fields:", len(missing_fields))

# ---- 2. Status distribution ----
status_counts = Counter(v.get("status") for v in order_log.values())
print("Status counts:", status_counts)

# ---- 3. Duplicate planet_item_id check ----
item_ids = [v["planet_item_id"] for v in order_log.values()]
dup_items = [i for i, c in Counter(item_ids).items() if c > 1]
print("Duplicate planet_item_id entries:", len(dup_items))

# ---- 4. Key format sanity check ----
bad_keys = [k for k in order_log if "_" not in k]
print("Badly formatted keys:", len(bad_keys))

# ---- 5. JSON safety check (round‑trip) ----
try:
    json.dumps(order_log)
    print("JSON serialization check: OK")
except Exception as e:
    print("JSON serialization check: FAILED", e)

# ---- 6. Print a sample ----
print("\nSample entries:")
for i, (k, v) in enumerate(order_log.items()):
    print(k, v)
    if i == 2:
        break


Total entries in order_log: 7726
Entries missing required fields: 0
Status counts: Counter({'submitted': 7715, 'failed': 11})
Duplicate planet_item_id entries: 1056
Badly formatted keys: 0
JSON serialization check: OK

Sample entries:
0_post {'pair_uid': 0, 'footprint_type': 'post', 'planet_item_id': '20220430_132000_1010', 'order_name': 'gedi_0_post_20220430_132000_1010', 'order_id': 'ee6e87d0-8b95-4f41-9dfd-d040cffd0356', 'status': 'submitted'}
0_pre {'pair_uid': 0, 'footprint_type': 'pre', 'planet_item_id': '20200423_142308_84_106e', 'order_name': 'gedi_0_pre_20200423_142308_84_106e', 'order_id': '238afcbe-1d1e-4107-980c-5a6a5f0c3d0d', 'status': 'submitted'}
1_pre {'pair_uid': 1, 'footprint_type': 'pre', 'planet_item_id': '20200920_125602_76_2257', 'order_name': 'gedi_1_pre_20200920_125602_76_2257', 'order_id': '03d1b3c0-05d6-464c-8481-1e3afd073948', 'status': 'submitted'}


In [22]:
# Create download_log only if it does not exist

DOWNLOAD_LOG_PATH = Path("planet_download_log.json")

if not DOWNLOAD_LOG_PATH.exists():
    with open(DOWNLOAD_LOG_PATH, "w") as f:
        json.dump({}, f, indent=2)
    print("Created new planet_download_log.json")
else:
    print("planet_download_log.json already exists")


Created new planet_download_log.json


In [57]:
#################  DOWNLOAD THE PLANET IMAGES  #################

ORDER_LOG_PATH = Path("planet_order_log.json")
DOWNLOAD_LOG_PATH = Path("planet_download_log.json")

DOWNLOAD_ROOT = drive_planet            #where the images will be stored)
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)


MAX_DOWNLOADS = 4                       #max number of downloads (start small and go up gradually)


# -------------------- LOAD LOGS --------------------

order_log = json.load(open(ORDER_LOG_PATH, "r"))

if DOWNLOAD_LOG_PATH.exists():
    download_log = json.load(open(DOWNLOAD_LOG_PATH, "r"))
else:
    download_log = {}


# -------------------- AUTH CHECK --------------------

auth = Auth.from_user_default_session()
if not auth.is_initialized():
    raise RuntimeError("Run `planet auth login` in a terminal first.")


# -------------------- ATOMIC JSON WRITE --------------------

def atomic_json_dump(obj, path):
    with tempfile.NamedTemporaryFile("w", dir=path.parent, delete=False) as tmp:
        json.dump(obj, tmp, indent=2)
        tmp_name = tmp.name
    os.replace(tmp_name, path)


# -------------------- DOWNLOAD FIRST N ORDERS --------------------

async def download_first_n_orders():

    downloaded_count = 0

    async with Session(auth=auth) as sess:
        orders = OrdersClient(sess)

        for key, info in order_log.items():

            # Stop after N successful downloads
            if downloaded_count >= MAX_DOWNLOADS:
                break

            # Only download successful, not-yet-downloaded orders
            if info.get("status") != "submitted":
                continue
            if key in download_log:
                continue

            order_id = info["order_id"]
            order_name = info.get("order_name", f"order_{key}")

            order_dir = DOWNLOAD_ROOT / order_name
            order_dir.mkdir(parents=True, exist_ok=True)

            print(f"⬇️ Downloading {downloaded_count + 1}/{MAX_DOWNLOADS}: {order_name}")

            # Wait until ready (safe if already completed)
            await orders.wait(order_id)

            # Download
            await orders.download_order(
                order_id,
                directory=str(order_dir)
            )

            # Record download
            download_log[key] = {
                "order_id": order_id,
                "order_name": order_name,
                "download_dir": str(order_dir),
                "downloaded": True,
                "timestamp": datetime.now(timezone.utc).isoformat()
            }

            atomic_json_dump(download_log, DOWNLOAD_LOG_PATH)

            downloaded_count += 1

        if downloaded_count == 0:
            print("⚠️ No eligible orders found to download.")
        else:
            print(f"✅ Downloaded {downloaded_count} images successfully.")


#  RUN 

await download_first_n_orders()


⬇️ Downloading 1/4: gedi_7107_pre_20200913_132353_1034
⬇️ Downloading 2/4: gedi_7127_pre_20200913_132352_1034
⬇️ Downloading 3/4: gedi_7151_pre_20200913_130001_02_2271
⬇️ Downloading 4/4: gedi_7191_pre_20200913_130001_02_2271
✅ Downloaded 4 images successfully.


In [58]:

#################  CHECK HOW MANY IMAGES REMAIN TO DOWNLOAD #################

import json
from pathlib import Path


# -------------------- PATHS --------------------

ORDER_LOG_PATH = Path("planet_order_log.json")
DOWNLOAD_LOG_PATH = Path("planet_download_log.json")


# -------------------- LOAD LOGS --------------------

order_log = json.load(open(ORDER_LOG_PATH, "r"))

if DOWNLOAD_LOG_PATH.exists():
    download_log = json.load(open(DOWNLOAD_LOG_PATH, "r"))
else:
    download_log = {}


# -------------------- ANALYSIS --------------------

# Orders that were successfully submitted (and thus should be downloadable)
submitted_orders = {
    key for key, info in order_log.items()
    if info.get("status") == "submitted"
}

# Orders that have already been downloaded
downloaded_orders = set(download_log.keys())

# Orders still pending download
pending_orders = sorted(submitted_orders - downloaded_orders)


# -------------------- REPORT --------------------

print("📦 ORDER / DOWNLOAD STATUS SUMMARY\n")

print(f"Total order records           : {len(order_log)}")
print(f"Orders successfully submitted : {len(submitted_orders)}")
print(f"Orders already downloaded     : {len(downloaded_orders)}")
print(f"Orders remaining to download: {len(pending_orders)}")

# Optional: show a few example keys
N_SHOW = 10
if pending_orders:
    print("\nExample pending order keys:")
    for key in pending_orders[:N_SHOW]:
        print(f"  - {key}")


📦 ORDER / DOWNLOAD STATUS SUMMARY

Total order records           : 7726
Orders successfully submitted : 7715
Orders already downloaded     : 7715
🟡 Orders remaining to download: 0


In [14]:
planet_all = pd.read_csv("planet_closest_images_complete.csv")
print("Images to order:", len(planet_all))



Images to order: 43772


In [48]:
######### PAIRS TO RETRY  #########
# Try to find another image to be downloaded if the first one had its access denied

# Get only the refused pairs according to the order_log

ORDER_LOG_PATH = Path("planet_order_log.json")

order_log = json.load(open(ORDER_LOG_PATH, "r"))

order_log_df = (
    pd.DataFrame.from_dict(order_log, orient="index")
    .reset_index(drop=True)
)

# Keep only access-denied failures
refused_df = order_log_df[
    (order_log_df["status"] == "failed") &
    (order_log_df["failure_category"] == "access_denied")
][["pair_uid", "footprint_type"]]

#print(f"Refused footprints: {len(refused_df)}")
#print(refused_df)


# Get the pairs and check which ones have extra potential images to retry the download

planet_df_all = planet_all.copy()  

# Sort and rank images per pair/footprint
planet_df_all = (
    planet_df_all
    .sort_values("days_from_target")
    .assign(
        rank_from_target=planet_df_all
        .groupby(["pair_uid", "footprint_type"])
        .cumcount()
    )
)

# Merge with the images that have a second image that satisfies the criteria
retry_df = (
    planet_df_all
    .merge(
        refused_df,
        on=["pair_uid", "footprint_type"],
        how="inner"
    )
    .query("rank_from_target == 3")
)

print(f"Retry images selected: {len(retry_df)}")
retry_df.head()



Retry images selected: 1


,pair_uid,footprint_type,planet_item_id,acquired_date,days_from_target,cloud_cover,overlap_pct,search_tier,rank_from_target
19,353,post,20200214_141106_32_106f,2020-02-14 14:11:06.328447+00:00,13,0.0,100.0,3,3


In [49]:
# Checking the images that do not have extra images to retry 
#(reality check to compare with the cell above)

missing_retries = (
    refused_df
    .merge(
        retry_df[["pair_uid", "footprint_type"]],
        on=["pair_uid", "footprint_type"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
)

print(f"Footprints with NO second image available: {len(missing_retries)}")
missing_retries.head()


Footprints with NO second image available: 10


,pair_uid,footprint_type,_merge
0,251,pre,left_only
1,267,pre,left_only
2,292,pre,left_only
3,330,pre,left_only
5,381,pre,left_only


In [50]:
# Adjust retry_df and check the head

retry_df = retry_df[
    [
        "pair_uid",
        "footprint_type",
        "planet_item_id",
        "days_from_target",
        "cloud_cover",
        "search_tier",
        "rank_from_target",
    ]
]

retry_df.head()
# Will have to change manually the input df in the orders cell!!!


,pair_uid,footprint_type,planet_item_id,days_from_target,cloud_cover,search_tier,rank_from_target
19,353,post,20200214_141106_32_106f,13,0.0,3,3


In [63]:
# AFTER RETRY
# Removing the pairs in which AT LEAST ONE footprint does not have an image

ORDER_LOG_PATH = Path("planet_order_log.json")

order_log = json.load(open(ORDER_LOG_PATH, "r"))

order_log_df = (
    pd.DataFrame.from_dict(order_log, orient="index")
    .reset_index(drop=True)
)


# For each pair_uid, check if ANY order failed
pair_failure_summary = (
    order_log_df
    .groupby("pair_uid")["status"]
    .apply(lambda s: (s == "failed").any())
    .reset_index(name="has_any_failed")
)

# pair_uid s to DROP
pairs_to_drop = pair_failure_summary.loc[
    pair_failure_summary["has_any_failed"],
    "pair_uid"
]

#print(f"Pairs to drop (any failure): {len(pairs_to_drop)}")
#print(pairs_to_drop.tolist())

pairs_planet = pairs[
    ~pairs["pair_uid"].isin(pairs_to_drop)
].copy()

#print("Number of pairs after Planet filtering:", len(pairs_planet))
#print("Removed pairs:", len(pairs) - len(pairs_planet))


# Export
#OUTPUT_GPKG = "GEDI_Pairs_Final_Planet.gpkg"

#pairs_planet.to_file(OUTPUT_GPKG, driver="GPKG")
#print(f"Saved filtered pairs to: {OUTPUT_GPKG}")


Saved filtered pairs to: GEDI_Pairs_Final_Planet.gpkg
